In [6]:
#LOAD ENV VARIABLES
from dotenv import load_dotenv

load_dotenv()

#Create an API client
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
model = "gemini-3.1-flash-lite"
#noinspection PyTypeChecker
def add_user_message(messages,text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(assistant_message)

def chat(messages,system=None,stop_sequences=None):

    params = {
        "model":model,
        "contents":messages,
        "config":types.GenerateContentConfig(
            stop_sequences=stop_sequences
        )
    }
    if system:
        params["config"] = types.GenerateContentConfig(
            system_instruction=system
        )

    message = client.models.generate_content(**params)
    return message.text

In [8]:
messages = []

add_user_message(messages,
                 "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Source": ["aws.ec2"],\n  "DetailType": ["EC2 State-change Notification"],\n  "Detail": {\n    "state": ["terminated"]\n  }\n}\n'

In [9]:
import json

json.loads(text.strip())

{'Source': ['aws.ec2'],
 'DetailType': ['EC2 State-change Notification'],
 'Detail': {'state': ['terminated']}}

In [23]:
messages = []
prompt = """
Generate three different sample AWS CLI commands.Each should be very different
"""

add_user_message(messages,prompt)
add_assistant_message(messages, "Here are all 3 commands in a single block without any comments:\n```bash")
text=chat(messages,stop_sequences=["```"])
text.strip()

'aws s3 cp my-local-file.txt s3://my-unique-bucket-name/uploads/\naws ec2 run-instances --image-id ami-0abcdef1234567890 --count 1 --instance-type t2.micro --key-name my-key-pair\naws iam create-user --user-name john_doe'

In [24]:
from IPython.display import Markdown
Markdown(text)


aws s3 cp my-local-file.txt s3://my-unique-bucket-name/uploads/
aws ec2 run-instances --image-id ami-0abcdef1234567890 --count 1 --instance-type t2.micro --key-name my-key-pair
aws iam create-user --user-name john_doe
